# Exploratory Data Analysis: Personal Expense Tracker

This notebook demonstrates static data analysis using **Matplotlib** and **Seaborn**.
It loads synthetic transaction data, cleans and normalizes records, and visualizes:
1. Category-wise spending with mean expenditure benchmark
2. Monthly time-series expense trends
3. Payment method distribution
4. Category budgets vs. actual spending with visual overspending indicators

In [ ]:
import sys
from pathlib import Path
import calendar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from expense_tracker.synthetic import generate_synthetic_transactions
from expense_tracker.analytics import clean_transaction_data, get_budget_status

print("Libraries and modules loaded.")

In [ ]:
# Generate reproducible synthetic transactions
raw_df = generate_synthetic_transactions(count=150, seed=42)
clean_df, errors = clean_transaction_data(raw_df)
expenses = clean_df[clean_df["type"] == "expense"].copy()
expenses["date"] = pd.to_datetime(expenses["date"])
expenses["month"] = expenses["date"].dt.strftime("%Y-%m")
print(f"Total expense rows: {len(expenses)}")
expenses.head()

In [ ]:
# 4-Panel Exploratory Visualizations
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Personal Expense Tracker — Static Exploratory Analysis (INR ₹)", fontsize=16, fontweight="bold")

# 1. Category spending
cat_summary = expenses.groupby("category")["amount"].sum().reset_index().sort_values("amount", ascending=False)
ax1 = axes[0, 0]
sns.barplot(data=cat_summary, x="category", y="amount", ax=ax1, palette="viridis")
ax1.set_title("Total Spending by Category", fontweight="bold")
ax1.set_ylabel("Amount (₹)")
ax1.tick_params(axis="x", rotation=35)
ax1.axhline(cat_summary["amount"].mean(), color="red", linestyle="--", label=f"Mean (₹{cat_summary['amount'].mean():,.0f})")
ax1.legend()

# 2. Monthly Trend
monthly_summary = expenses.groupby("month")["amount"].sum().reset_index().sort_values("month")
ax2 = axes[0, 1]
sns.lineplot(data=monthly_summary, x="month", y="amount", marker="o", ax=ax2, color="#1f77b4", linewidth=2.5)
ax2.set_title("Monthly Spending Trend", fontweight="bold")
ax2.set_ylabel("Expenses (₹)")
ax2.tick_params(axis="x", rotation=25)

# 3. Payment Method
pay_summary = expenses.groupby("payment_method")["amount"].sum().reset_index().sort_values("amount", ascending=False)
ax3 = axes[1, 0]
sns.barplot(data=pay_summary, x="payment_method", y="amount", ax=ax3, palette="mako")
ax3.set_title("Spending by Payment Method", fontweight="bold")
ax3.set_ylabel("Amount (₹)")

# 4. Budget vs Actual
budgets = {"Groceries": 350.0, "Food & Dining": 200.0, "Transport": 150.0, "Utilities": 100.0}
b_statuses = get_budget_status(expenses, budgets)
b_df = pd.DataFrame(b_statuses)
ax4 = axes[1, 1]
if not b_df.empty:
    plot_df = b_df.melt(id_vars=["category", "is_over"], value_vars=["spent", "budget"], var_name="Metric", value_name="Amount")
    sns.barplot(data=plot_df, x="category", y="Amount", hue="Metric", ax=ax4, palette={"spent": "#e74c3c", "budget": "#2ecc71"})
    ax4.set_title("Budget vs. Actual (Red = Spent, Green = Budget)", fontweight="bold")
    ax4.set_ylabel("Amount (₹)")

plt.tight_layout()
plt.show()